# §2.4.3 — 온도를 바꾸면 무엇이 바뀌고 무엇이 안 바뀌는가

> 딥러닝 교재 · 1부 2장 4절 4항 (🐍)
> 선행: §2.3.6(손실이 0으로 안 가는 것) · §2.4.1(평행이동 불변성) · §2.4.2(야코비안) · §2.4.3(온도와 두 극한)

## 이 노트북이 답하는 질문

1. **정확도는 정말 $T$ 에 무관한가?** §2.4.3 3절 (가)의 주장을 수치로 확인한다 — 근사가 아니라 **정확히** 같아야 한다.
2. **그러면 무엇이 바뀌는가?** 엔트로피 · NLL · 보정 오차를 함께 훑는다.
3. **$T = 1$ 이 최선인가?** 아니라면 그 사실이 무엇을 뜻하는가.
4. **§2.4.3이 계산한 수렴 속도가 맞는가?** $T \to 0$ 은 지수적으로, $T \to \infty$ 는 $1/T$ 로.

**예상 실행 시간** CPU 단일 코어 약 20초.

참 조건부 분포 $\eta(x)$ 를 아는 합성 자료를 쓴다. 그래야 **"모형의 확률이 참값에 얼마나 가까운가"**를 직접 잴 수 있다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260808
K, D     = 3, 2          # 클래스 수, 입력 차원
N_TRAIN  = 1000          # 훈련 표본 수 (작을수록 과확신이 심해진다)
STEPS    = 2500
WIDTH    = 32
N_TEST   = 20_000
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────
if FAST:
    N_TEST, STEPS = 8000, 1500

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_2_4_4_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | K={K} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 자료 — 참 $\eta(x)$ 를 안다

$$x \sim \mathcal{N}(0, I_2), \qquad \eta(x) = \mathrm{softmax}(A^{\top}x + \beta), \qquad y \sim \mathrm{Categorical}(\eta(x))$$

참 조건부 분포가 폐형식이므로 **베이즈 하한도 계산할 수 있다**(§2.3.6 1절).

$$R^{\ast} = \mathbb{E}_x\big[H(\eta(x))\big]$$

이 값이 이후 모든 NLL의 기준선이 된다.

In [ ]:
def log_softmax(z):
    # §2.3.5의 규약: 소프트맥스를 만든 뒤 로그를 취하지 않는다
    m = z.max(-1, keepdims=True)
    s = z - m
    return s - np.log(np.exp(s).sum(-1, keepdims=True))

_g = np.random.default_rng(3)
A_TRUE = _g.normal(0, 1.6, (D, K))
B_TRUE = _g.normal(0, 0.4, K)

def eta(x):
    return np.exp(log_softmax(np.asarray(x) @ A_TRUE + B_TRUE))

def sample(n, g):
    x = g.normal(0, 1, (n, D))
    p = eta(x)
    u = g.random(n)[:, None]
    return x, (u > np.cumsum(p, axis=1)).sum(1)

x_tr, y_tr = sample(N_TRAIN, np.random.default_rng(SEED))
x_te, y_te = sample(N_TEST,  np.random.default_rng(SEED+1))
Y_te = np.zeros((len(y_te), K)); Y_te[np.arange(len(y_te)), y_te] = 1.0

E_TE = eta(x_te)
R_STAR = float(-(E_TE*np.log(E_TE)).sum(1).mean())
ACC_BAYES = float((E_TE.argmax(1) == y_te).mean())
print(f"훈련 {N_TRAIN}개 · 시험 {N_TEST:,}개, 클래스 {K}개")
print(f"베이즈 하한  R* = {R_STAR:.4f}   (무작위 추측은 log K = {np.log(K):.4f})")
print(f"베이즈 정확도 = {ACC_BAYES:.4f}")

---
## 2. 모형 학습

§2.3.2의 기울기 $q - \mathbf{y}$ 를 그대로 쓰고, §2.3.5의 규약대로 `log_softmax` 만 사용한다.

In [ ]:
def init_net(L, w, g):
    dims = [D] + [w]*L + [K]; Ws, bs = [], []
    for i in range(len(dims)-1):
        Ws.append(g.normal(0, np.sqrt(2/dims[i]), (dims[i], dims[i+1])))
        bs.append(np.zeros(dims[i+1]))
    return Ws, bs

def fwd(x, Ws, bs):
    a = np.asarray(x); acts=[a]; pre=[]
    for i in range(len(Ws)-1):
        z = a @ Ws[i] + bs[i]; pre.append(z); a = np.maximum(z,0); acts.append(a)
    return a @ Ws[-1] + bs[-1], acts, pre

def train(Ws, bs, x, y, steps, lr=5e-3):
    st = {k: [np.zeros_like(p) for p in (Ws if k[-1]=='W' else bs)]
          for k in ('mW','vW','mb','vb')}
    n = len(x); Y = np.zeros((n, K)); Y[np.arange(n), y] = 1.0
    for t in range(1, steps+1):
        z, acts, pre = fwd(x, Ws, bs)
        g = (np.exp(log_softmax(z)) - Y)/n          # §2.3.2
        gW=[None]*len(Ws); gb=[None]*len(bs)
        gW[-1] = acts[-1].T @ g; gb[-1] = g.sum(0); d = g @ Ws[-1].T
        for i in range(len(Ws)-2, -1, -1):
            d = d*(pre[i] > 0); gW[i] = acts[i].T @ d; gb[i] = d.sum(0)
            if i > 0:
                d = d @ Ws[i].T
        for i in range(len(Ws)):
            for (p, gp, mk, vk) in ((Ws[i],gW[i],'mW','vW'), (bs[i],gb[i],'mb','vb')):
                st[mk][i] = 0.9*st[mk][i] + 0.1*gp
                st[vk][i] = 0.999*st[vk][i] + 0.001*gp**2
                p -= lr*(st[mk][i]/(1-0.9**t))/(np.sqrt(st[vk][i]/(1-0.999**t)) + 1e-8)

W, b = init_net(3, WIDTH, np.random.default_rng(SEED+5))
train(W, b, x_tr, y_tr, STEPS)
Z_TE = fwd(x_te, W, b)[0]                      # 시험 로짓
z_tr = fwd(x_tr, W, b)[0]

acc_tr = float((z_tr.argmax(1) == y_tr).mean())
acc_te = float((Z_TE.argmax(1) == y_te).mean())
print(f"훈련 정확도 {acc_tr:.4f}   시험 정확도 {acc_te:.4f}   (베이즈 {ACC_BAYES:.4f})")
print(f"-> 훈련과 시험의 격차가 과확신의 원인이다 (§1.1.2)")

---
## 3. 온도를 훑는다

다섯 가지를 함께 잰다.

| 지표 | 무엇을 보는가 |
|---|---|
| 정확도 | $\arg\max$ — §2.4.3 3절 (가)에 따르면 **$T$ 에 무관해야 한다** |
| 평균 엔트로피 | 확신의 정도 — §2.4.3 3절 (나)에 따르면 **단조 증가** |
| 시험 NLL | 실제 목적함수 |
| ECE | 보정 오차 — 확신도와 실제 정확도의 차이 |
| $\mathbb{E}_x[\mathrm{KL}(\eta \,\|\, q)]$ | **참값과의 거리.** 참 $\eta$ 를 알기에 잴 수 있다 |

In [ ]:
def metrics(T, z=Z_TE, y=y_te, Y=Y_te, e=E_TE):
    lq = log_softmax(z/T); q = np.exp(lq)
    acc = float((q.argmax(1) == y).mean())
    nll = float(-(Y*lq).sum(1).mean())
    ent = float(-(q*lq).sum(1).mean())
    conf = q.max(1); corr = (q.argmax(1) == y).astype(float)
    edges = np.linspace(1.0/K, 1.0, 16); ece = 0.0
    for i in range(len(edges)-1):
        m = (conf >= edges[i]) & (conf < edges[i+1] + (1e-9 if i == len(edges)-2 else 0))
        if m.sum() > 0:
            ece += m.mean()*abs(conf[m].mean() - corr[m].mean())
    kl = float((e*(np.log(e + 1e-300) - lq)).sum(1).mean())
    return dict(acc=acc, nll=nll, ent=ent, ece=ece, kl=kl, conf=float(conf.mean()))

TS = np.exp(np.linspace(np.log(0.2), np.log(50), 60 if not FAST else 30))
M = {k: np.array([metrics(T)[k] for T in TS]) for k in ('acc','nll','ent','ece','kl','conf')}

# §2.4.3 3절 (가): 정확도는 정확히 상수여야 한다
assert np.allclose(M['acc'], M['acc'][0], atol=0, rtol=0), "정확도가 T에 의존한다"
print(f"정확도: 모든 T에서 정확히 {M['acc'][0]:.6f}  (60개 온도에서 비트 단위로 동일)")

i1 = int(np.argmin(np.abs(TS - 1.0)))
iN = int(np.argmin(M['nll'])); iE = int(np.argmin(M['ece'])); iK = int(np.argmin(M['kl']))
print(f"\n           T        NLL      엔트로피     ECE      KL(η‖q)   평균확신도")
for nm, i in [("T = 1", i1), ("NLL 최소", iN), ("ECE 최소", iE), ("KL 최소", iK)]:
    print(f"  {nm:>9} {TS[i]:6.2f}   {M['nll'][i]:7.4f}   {M['ent'][i]:7.4f}  "
          f"{M['ece'][i]:7.4f}   {M['kl'][i]:7.4f}    {M['conf'][i]:7.4f}")
print(f"\n  베이즈 하한 R* = {R_STAR:.4f}")
print(f"  실제 정확도  = {M['acc'][0]:.4f}  <- 평균 확신도와 비교할 것")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12.4, 3.2))
panels = [('acc', lab('정확도','accuracy'), CB[5], None),
          ('ent', lab('평균 엔트로피','mean entropy'), CB[3], np.log(K)),
          ('nll', lab('시험 NLL','test NLL'), CB[4], R_STAR),
          ('ece', lab('ECE','ECE'), CB[6], 0.0)]
for ax, (key, ttl, c, ref) in zip(axes, panels):
    ax.semilogx(TS, M[key], color=c, lw=1.8)
    ax.axvline(1.0, color=CB[0], ls=':', lw=1.2)
    if ref is not None:
        ax.axhline(ref, color=CB[0], ls='--', lw=1.0)
    if key == 'acc':
        ax.set_ylim(0, 1)
        ax.text(0.3, 0.15, lab('완전히 평평하다','exactly flat'), fontsize=8, color=CB[5])
    if key in ('nll','ece'):
        j = int(np.argmin(M[key]))
        ax.plot(TS[j], M[key][j], '*', ms=13, color=CB[1])
    ax.set_xlabel(lab('온도 $T$','temperature $T$'))
    ax.set_title(ttl, fontsize=10)
fig.suptitle(lab('$T$ 는 정확도를 전혀 바꾸지 않으면서 나머지 모두를 바꾼다',
                 'temperature changes everything except accuracy'), y=1.04, fontsize=10)
show('temperature_sweep')

> ### 이것이 이 절의 결론이다
>
> 왼쪽 패널이 **완전히 평평하다.** 60개의 온도에서 정확도가 비트 단위로 같다 — §2.4.3 3절 (가)가 증명한 대로
> $\arg\max$ 가 $T$ 에 무관하기 때문이다.
>
> 나머지 셋은 크게 움직인다. **정확도만 보는 평가는 이 변화를 전혀 감지하지 못한다.**
>
> 그리고 NLL의 최소가 $T = 1$ 이 아니다. **모형이 스스로 내놓는 확률이 최선이 아니라는 뜻**이며,
> 이것이 §2.4.5가 굵게 쓸 문장이다 — **소프트맥스 출력은 확률이지만 보정된 확률은 아니다.**

---
## 4. 신뢰도 그림

확신도별로 묶어 **그 구간의 실제 정확도**를 그린다. 보정이 잘 되었다면 대각선 위에 놓인다.

In [ ]:
def reliability(T, nb=12):
    q = np.exp(log_softmax(Z_TE/T))
    conf = q.max(1); corr = (q.argmax(1) == y_te).astype(float)
    edges = np.linspace(1.0/K, 1.0, nb+1)
    cs, accs, ws = [], [], []
    for i in range(nb):
        hi = edges[i+1] + (1e-9 if i == nb-1 else 0)
        m = (conf >= edges[i]) & (conf < hi)
        if m.sum() > 30:
            cs.append(conf[m].mean()); accs.append(corr[m].mean()); ws.append(m.mean())
    return np.array(cs), np.array(accs), np.array(ws)

fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.4), sharey=True)
for ax, (T, ttl) in zip(axes, [(1.0, lab('$T$ = 1 (원래 모형)','$T$ = 1')),
                               (TS[iN], lab(f'$T$ = {TS[iN]:.2f} (NLL 최소)', f'$T$ = {TS[iN]:.2f}')),
                               (20.0, lab('$T$ = 20 (과하게 뜨겁다)','$T$ = 20'))]):
    c, a, w = reliability(T)
    ax.plot([1/K, 1], [1/K, 1], color=CB[0], ls='--', lw=1.2, label=lab('완전 보정','perfect'))
    ax.plot(c, a, 'o-', ms=5, color=CB[5])
    ax.set_xlim(1/K - 0.03, 1.03); ax.set_ylim(1/K - 0.03, 1.03)
    ax.set_xlabel(lab('예측 확신도','confidence')); ax.set_title(ttl, fontsize=10)
axes[0].set_ylabel(lab('실제 정확도','observed accuracy')); axes[0].legend(fontsize=8)
fig.suptitle(lab('대각선 아래 = 과확신 · 위 = 과소확신',
                 'below diagonal = overconfident'), y=1.04, fontsize=10)
show('reliability')

print("      T     평균 확신도   실제 정확도    차이")
for T in [1.0, TS[iN], 20.0]:
    q = np.exp(log_softmax(Z_TE/T))
    cf = float(q.max(1).mean()); ac = float((q.argmax(1) == y_te).mean())
    print(f"  {T:6.2f}   {cf:10.4f}   {ac:10.4f}   {cf-ac:+8.4f}")

---
## 5. §2.4.3의 수렴 속도 검증

$$T \to 0: \quad 1 - \max_k q^{(T)}_k = O\big(e^{-\Delta/T}\big), \qquad
T \to \infty: \quad \max_k\Big|q^{(T)}_k - \tfrac1K\Big| = O(1/T)$$

$\Delta$ 는 최댓값과 두 번째 값의 격차다. **차가워지는 것은 지수적으로 빠르고 뜨거워지는 것은 느리다.**

In [ ]:
zs = np.sort(Z_TE, axis=1)
DELTA = float(np.median(zs[:, -1] - zs[:, -2]))
print(f"로짓 격차 Δ 의 중앙값 = {DELTA:.3f}")

T_cold = np.exp(np.linspace(np.log(0.02), np.log(1.0), 25))
T_hot  = np.exp(np.linspace(np.log(1.0), np.log(3000), 25))
cold = np.array([float((1 - np.exp(log_softmax(Z_TE/T)).max(1)).mean()) for T in T_cold])
hot  = np.array([float(np.abs(np.exp(log_softmax(Z_TE/T)) - 1.0/K).max(1).mean()) for T in T_hot])

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
axes[0].semilogy(1.0/T_cold, cold, 'o-', ms=4, color=CB[5], label=lab('실측','observed'))
axes[0].semilogy(1.0/T_cold, np.exp(-DELTA/T_cold)*cold[0]/np.exp(-DELTA/T_cold[0]),
                 'k--', lw=1.2, label=lab(r'$e^{-\Delta/T}$', r'$e^{-\Delta/T}$'))
axes[0].set_xlabel(r'$1/T$'); axes[0].set_ylabel(lab(r'$1 - \max_k q_k$', r'$1-\max_k q_k$'))
axes[0].set_title(lab('$T \\to 0$: 지수적', 'cooling: exponential'), fontsize=10)
axes[0].legend(fontsize=8)

axes[1].loglog(T_hot, hot, 'o-', ms=4, color=CB[3], label=lab('실측','observed'))
axes[1].loglog(T_hot, hot[0]*T_hot[0]/T_hot, 'k--', lw=1.2, label=lab(r'$1/T$', r'$1/T$'))
axes[1].set_xlabel('$T$'); axes[1].set_ylabel(lab(r'$\max_k |q_k - 1/K|$', r'$\max_k|q_k-1/K|$'))
axes[1].set_title(lab('$T \\to \\infty$: $1/T$', 'heating: $1/T$'), fontsize=10)
axes[1].legend(fontsize=8)
show('convergence_rates')

s_hot, _ = np.polyfit(np.log(T_hot[-12:]), np.log(hot[-12:]), 1)
print(f"뜨거운 쪽 로그-로그 기울기 = {s_hot:+.3f}  (이론 -1)")

---
## 6. 과확신은 어디서 오는가

$T^{\ast}$ 가 1보다 크다는 것은 모형이 **과확신**이라는 뜻이다. 그렇다면 무엇이 그것을 만드는가.
훈련 표본 수를 바꿔 확인한다.

In [ ]:
NS = [300, 1000, 3000] + ([] if FAST else [10_000])
print("   n     훈련정확도  시험정확도   T*     NLL(T=1)   NLL(T*)    R*")
for n_ in NS:
    xt, yt = sample(n_, np.random.default_rng(SEED))
    Wn, bn = init_net(3, WIDTH, np.random.default_rng(SEED+5))
    train(Wn, bn, xt, yt, STEPS)
    zt = fwd(x_te, Wn, bn)[0]
    nlls = np.array([float(-(Y_te*log_softmax(zt/T)).sum(1).mean()) for T in TS])
    j = int(np.argmin(nlls)); j1 = int(np.argmin(np.abs(TS-1.0)))
    tr = float((fwd(xt, Wn, bn)[0].argmax(1) == yt).mean())
    te = float((zt.argmax(1) == y_te).mean())
    print(f"{n_:>6}   {tr:8.3f}   {te:8.3f}  {TS[j]:6.2f}   {nlls[j1]:8.3f}   {nlls[j]:7.3f}   {R_STAR:.3f}")
print("\n-> 훈련·시험 정확도의 격차가 줄면 T* 가 1에 가까워진다.")
print("   과확신은 모형의 결함이 아니라 §1.1.2의 낙관 편향이 확률에 나타난 형태다.")

---
## 7. 자기 점검

1. 3절에서 정확도가 **완전히** 평평했다. §2.4.3의 어느 성질 때문인가? 그 성질이 깨지는 결정 규칙이 있는가?
2. NLL 최소 $T$ 와 ECE 최소 $T$ 가 다를 수 있다. **어느 것을 써야 하는가?**
3. 5절에서 뜨거운 쪽 수렴이 느렸다. 이것이 **온도 보정의 실무에 어떤 함의**를 갖는가?
4. 6절에서 $n$ 을 아주 크게 하면 $T^{\ast} \to 1$ 이 되겠는가? 그래도 남는 문제는 없는가?

In [ ]:
# 자기 점검 1의 확인 — 표집으로 결정하면 T가 결과를 바꾼다
print("결정 규칙에 따라 T의 영향이 달라진다 (시험 자료 앞 5000개)")
print("      T     argmax 정확도   표집 정확도")
gs = np.random.default_rng(0)
sub = slice(0, 5000)
for T in [0.2, 1.0, 5.0, 50.0]:
    q = np.exp(log_softmax(Z_TE[sub]/T))
    a_arg = float((q.argmax(1) == y_te[sub]).mean())
    u = gs.random(len(q))[:, None]
    ysamp = (u > np.cumsum(q, axis=1)).sum(1)
    a_smp = float((ysamp == y_te[sub]).mean())
    print(f"  {T:6.2f}   {a_arg:11.4f}   {a_smp:11.4f}")
print("\n-> argmax 는 T에 완전히 무관하지만 표집은 그렇지 않다 (§2.4.3 4절).")
print("   32장의 생성에서 T가 결과를 바꾸는 것은 결정 규칙이 표집이기 때문이다.")

---
## 8. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `N_TRAIN` | 0절 | 1000 | 작을수록 과확신이 심해지고 $T^{\ast}$ 가 커진다 |
| `STEPS` | 0절 | 2500 | 오래 학습할수록 과확신 (§2.3.2 (b) — 로짓이 자란다) |
| `WIDTH` | 0절 | 32 | 용량 |
| `TS` | 3절 | 0.2 ~ 50 | 온도 격자 |
| `A_TRUE` 의 크기 | 1절 | 1.6 | 참 문제의 난이도. 크면 $\eta$ 가 뾰족해져 $R^{\ast}$ 가 작아진다 |

**권하는 첫 실험** — `STEPS` 를 2500에서 10000으로 늘리십시오. **정확도는 거의 그대로인데 $T^{\ast}$ 가 크게 오릅니다.**
더 오래 학습하는 것이 §2.3.2 (b)에서 본 대로 로짓을 키우고, 그것이 곧 과확신입니다.
**정확도만 보고 있으면 이 변화가 전혀 보이지 않습니다.**

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")